# SmoothQuant W8A8 — Fixed for Colab

**What this notebook does:**
- Path 1 — Real W8A8 INT8: load the official `mit-han-lab/opt-125m-smoothquant` checkpoint
  using `Int8OPTForCausalLM` (requires `torch_int` CUDA extension to compile).
- Path 2 — Fake W8A8: apply `smooth_lm` + `quantize_opt` on `facebook/opt-125m`
  (works without `torch_int`; simulates quantization in fp for accuracy analysis).

**Dependency chain (the full picture):**
```
smoothquant
  ├── smoothquant.smooth      → only needs torch + transformers  ✓ always works
  ├── smoothquant.fake_quant  → only needs torch + transformers  ✓ always works
  └── smoothquant.opt         → needs torch_int (CUDA ext, compile from source)
         └── torch_int        → https://github.com/Guangxuan-Xiao/torch-int
```

**Run order:**
1. Run Cell 1 (installs everything including torch_int compilation)
2. **Runtime → Restart session**
3. Run Cells 2 onward

## Cell 1 — Install All Dependencies
⚠️ **Restart the runtime after this cell completes.**

In [1]:
import os, subprocess, sys

# ── 1. Core Python deps ─────────────────────────────────────────────────────
# Pin transformers to 4.36.x — smoothquant's OPT wrappers were written against
# this version. Newer versions restructured OPT internals and break the import.
!pip -q install transformers==4.36.2 accelerate datasets \
    sentencepiece safetensors ninja

# ── 2. Clone smoothquant ─────────────────────────────────────────────────────
SQ_DIR = "/content/smoothquant"
if not os.path.isdir(SQ_DIR):
    !git clone -q https://github.com/mit-han-lab/smoothquant {SQ_DIR}
else:
    print("smoothquant already cloned.")
!pip -q install -e {SQ_DIR}

# ── 3. torch_int — the hidden dependency that smoothquant.opt actually needs ─
# smoothquant/opt.py line 15 does:
#   from torch_int.nn.linear import W8A8BFP32OFP32Linear, ...
# torch_int is a separate MIT HAN Lab CUDA extension repo.
# It must be compiled from source — there are no pre-built wheels.
print("\nCompiling torch_int CUDA extension (this takes ~3-5 min)...")
TI_DIR = "/content/torch-int"
if not os.path.isdir(TI_DIR):
    !git clone -q https://github.com/Guangxuan-Xiao/torch-int {TI_DIR}
else:
    print("torch-int already cloned.")

# Set CUDA arch list to cover all Colab GPU generations:
# T4=7.5, A100=8.0, V100=7.0, L4/A10G=8.6
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.0;7.5;8.0;8.6;8.9;9.0"
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", TI_DIR, "--no-build-isolation"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("torch_int compiled and installed OK")
else:
    print("torch_int compilation FAILED (see below). Path 1 will be skipped.")
    print(result.stderr[-3000:])  # show last 3k chars of error

# ── 4. Verify files ──────────────────────────────────────────────────────────
required = [
    f"{SQ_DIR}/smoothquant/opt.py",
    f"{SQ_DIR}/smoothquant/smooth.py",
    f"{SQ_DIR}/smoothquant/fake_quant.py",
    f"{SQ_DIR}/act_scales/opt-125m.pt",
]
missing = [p for p in required if not os.path.exists(p)]
print("Missing files:", missing if missing else "none")

print("\n" + "="*60)
print("RESTART THE RUNTIME NOW → Runtime → Restart session")
print("Then run from Cell 2 onward.")
print("="*60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 53.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
  Preparing metadata (setup.py) ... done

Compiling torch_int CUDA extension (this takes ~3-5 min)...
torch_int compilation FAILED (see below). Path 1 will be skipped.
    error: subprocess-exited-with-error
    
    × python setup.py develop did not run successfully.
    │ exit code: 1
    ╰─> See above 

In [2]:
# Download the Pile val backup that generate_act_scales.py expects
!wget -q --show-progress -O /content/val.jsonl.zst \
    "https://huggingface.co/datasets/mit-han-lab/pile-val-backup/resolve/main/val.jsonl.zst"

!python /content/smoothquant/examples/generate_act_scales.py \
    --model-name facebook/opt-125m \
    --output-path /content/smoothquant/act_scales/opt-125m.pt \
    --dataset-path /content/val.jsonl.zst \
    --num-samples 512 \
    --seq-len 512

import os
size = os.path.getsize("/content/smoothquant/act_scales/opt-125m.pt")
print(f"act_scales size: {size:,} bytes")  # should be ~500KB+

/content/val.jsonl. 100%[===================>] 449.09M   257MB/s    in 1.8s    
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
tokenizer_config.json: 100% 685/685 [00:00<00:00, 4.53MB/s]
config.json: 100% 651/651 [00:00<00:00, 4.71MB/s]
vocab.json: 899kB [00:00, 53.1MB/s]
merges.txt: 456kB [00:00, 52.6MB/s]
special_tokens_map.json: 100% 441/441 [00:00<00:00, 2.67MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is depre

## Cell 2 — Imports
Run after restarting the runtime.

In [3]:
import json, os, sys, importlib, time
from pathlib import Path

import torch
from transformers import AutoTokenizer
from transformers.models.opt.modeling_opt import OPTForCausalLM

SQ_DIR = "/content/smoothquant"
TI_DIR = "/content/torch-int"

# Belt-and-suspenders: make sure both repos are on the path after restart
for p in [SQ_DIR, TI_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

# ── smooth_lm and fake_quant do NOT need torch_int — always import these ─────
from smoothquant.smooth import smooth_lm
from smoothquant.fake_quant import quantize_opt
print("smooth_lm, quantize_opt imported OK")

# ── Int8OPTForCausalLM needs torch_int — try but don't crash if absent ───────
HAS_TORCH_INT = False
try:
    import torch_int  # noqa: F401
    from smoothquant.opt import Int8OPTForCausalLM
    HAS_TORCH_INT = True
    print("torch_int found — Path 1 (real INT8) available")
except ImportError as e:
    print(f"torch_int not available ({e})")
    print("Path 1 (real INT8 via Int8OPTForCausalLM) will be skipped.")
    print("Path 2 (fake W8A8) is unaffected.")

print(f"\ntorch        : {torch.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"HAS_TORCH_INT: {HAS_TORCH_INT}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


smooth_lm, quantize_opt imported OK
torch_int not available (No module named 'torch_int._CUDA')
Path 1 (real INT8 via Int8OPTForCausalLM) will be skipped.
Path 2 (fake W8A8) is unaffected.

torch        : 2.10.0+cu128
CUDA         : True
GPU          : Tesla T4
HAS_TORCH_INT: False


## Cell 3 — Config

In [4]:
# SmoothQuant's smooth_lm / quantize_opt / Int8OPTForCausalLM are all
# OPT-architecture specific. Do NOT use TinyLlama / LLaMA / Mistral here.
MODEL_ID          = "facebook/opt-125m"
SMOOTHQUANT_HF_ID = "mit-han-lab/opt-125m-smoothquant"  # official INT8 export
ACT_SCALES_PATH   = Path(SQ_DIR) / "act_scales" / "opt-125m.pt"
OUTPUT_DIR        = Path("/content/quant_outputs/smoothquant_w8a8")
PROMPT            = "Quantization reduces inference memory by"
MAX_NEW_TOKENS    = 64
ALPHA             = 0.5  # SmoothQuant migration strength; paper default
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE             = torch.float16 if torch.cuda.is_available() else torch.float32

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def bytes_to_gib(n):
    return round(n / (1024 ** 3), 4)

def dir_size_gib(path):
    path = Path(path)
    if not path.exists(): return 0.0
    return bytes_to_gib(sum(p.stat().st_size for p in path.rglob("*") if p.is_file()))

def save_metrics(path, payload):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))

print(f"MODEL_ID      : {MODEL_ID}")
print(f"device/dtype  : {DEVICE} / {DTYPE}")
print(f"act_scales    : {ACT_SCALES_PATH}  exists={ACT_SCALES_PATH.exists()}")
print(f"output_dir    : {OUTPUT_DIR}")

MODEL_ID      : facebook/opt-125m
device/dtype  : cuda / torch.float16
act_scales    : /content/smoothquant/act_scales/opt-125m.pt  exists=True
output_dir    : /content/quant_outputs/smoothquant_w8a8


## Cell 4 — Tokenizer

In [5]:
# Must use the OPT tokenizer (GPT-2 BPE). The HF smoothquant checkpoint
# uses the same vocab as facebook/opt-125m so we load once and reuse both paths.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
print(f"Tokenizer: {MODEL_ID}  vocab_size={tokenizer.vocab_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenizer: facebook/opt-125m  vocab_size=50265


## Cell 5 — Path 1: Real INT8 via Official HF Checkpoint

Loads `mit-han-lab/opt-125m-smoothquant` using `Int8OPTForCausalLM`.
**Requires `torch_int` compiled in Cell 1.** Skipped automatically if not available.

The checkpoint was exported by the paper authors using their `export_int8_model.py` script
— weights are already int8 on disk.

In [6]:
if not HAS_TORCH_INT:
    print("[Path 1 SKIPPED] torch_int not available.")
    print("If you need this path: check Cell 1 torch_int compilation output for errors.")
    print("Common cause: PyTorch/CUDA version mismatch. Run Cell 1 again with GPU runtime.")
else:
    path1_dir = OUTPUT_DIR / "official_int8_model"

    print(f"Loading {SMOOTHQUANT_HF_ID} ...")
    int8_model = Int8OPTForCausalLM.from_pretrained(SMOOTHQUANT_HF_ID, device_map="auto")
    int8_model.eval()

    inputs = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = int8_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    print("\n[Path 1 generation]")
    print(tokenizer.decode(out[0], skip_special_tokens=True))

    int8_model.save_pretrained(path1_dir)
    tokenizer.save_pretrained(path1_dir)

    save_metrics(OUTPUT_DIR / "metrics_path1.json", {
        "method":            "smoothquant_w8a8_real_int8",
        "model_id":          MODEL_ID,
        "smoothquant_hf_id": SMOOTHQUANT_HF_ID,
        "artifact_dir":      str(path1_dir),
        "artifact_size_gib": dir_size_gib(path1_dir),
    })

[Path 1 SKIPPED] torch_int not available.
If you need this path: check Cell 1 torch_int compilation output for errors.
Common cause: PyTorch/CUDA version mismatch. Run Cell 1 again with GPU runtime.


## Cell 6 — Path 2: Smooth + Fake W8A8 (always works, no torch_int needed)

1. Load `facebook/opt-125m` in fp16.
2. Apply `smooth_lm` — absorbs activation outliers into weights via per-channel scaling.
3. Apply `quantize_opt` — wraps linears with fake W8A8 (quant→dequant in fp, no real int8 kernel).
4. Run generation and save scales.

`smooth_lm` and `quantize_opt` import only from torch/transformers — **no torch_int**.

In [7]:
if not ACT_SCALES_PATH.exists():
    raise FileNotFoundError(
        f"{ACT_SCALES_PATH} not found.\n"
        f"Run: ls {SQ_DIR}/act_scales/  to debug."
    )

print(f"Loading {MODEL_ID} ...")
model = OPTForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
)
model.eval()

act_scales = torch.load(ACT_SCALES_PATH, map_location="cpu")
print(f"act_scales loaded  ({len(act_scales)} keys)")

smooth_lm(model, act_scales, alpha=ALPHA)
print(f"smooth_lm applied  (alpha={ALPHA})")

fake_w8a8_model = quantize_opt(model)
print("quantize_opt (fake W8A8) applied")

infer_device = next(fake_w8a8_model.parameters()).device
inputs = tokenizer(PROMPT, return_tensors="pt").to(infer_device)
with torch.no_grad():
    out = fake_w8a8_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
print("\n[Path 2 generation]")
print(tokenizer.decode(out[0], skip_special_tokens=True))

scales_out = OUTPUT_DIR / "smoothquant_scales_and_config.pt"
torch.save({"model_id": MODEL_ID, "alpha": ALPHA, "act_scales": act_scales}, scales_out)
print(f"\nSaved scales → {scales_out}")

save_metrics(OUTPUT_DIR / "metrics_path2.json", {
    "method":     "smoothquant_w8a8_fake_quant",
    "model_id":   MODEL_ID,
    "alpha":      ALPHA,
    "act_scales": str(ACT_SCALES_PATH),
    "scales_out": str(scales_out),
})

Loading facebook/opt-125m ...
act_scales loaded  (73 keys)
smooth_lm applied  (alpha=0.5)
quantize_opt (fake W8A8) applied

[Path 2 generation]
Quantization reduces inference memory by using a quantum computing technique. The quantum computing technique is a quantum computer that uses a quantum computer to compute a quantum number. The quantum computing technique is a quantum computer that uses a quantum computer to compute a quantum number. The quantum computing technique is a quantum computer that uses a quantum computer to compute a quantum number. The

Saved scales → /content/quant_outputs/smoothquant_w8a8/smoothquant_scales_and_config.pt
{
  "method": "smoothquant_w8a8_fake_quant",
  "model_id": "facebook/opt-125m",
  "alpha": 0.5,
  "act_scales": "/content/smoothquant/act_scales/opt-125m.pt",
  "scales_out": "/content/quant_outputs/smoothquant_w8a8/smoothquant_scales_and_config.pt"
}


## Cell 7 — (Optional) Real INT8 Export via Official Script

Produces real INT8 artifacts using Pile val set for calibration. Requires `torch_int`.
Download the Pile val file (~1.5 GB) first, then uncomment.

In [8]:
# !wget -q -O /content/val.jsonl.zst \
#     https://the-eye.eu/public/AI/pile/val.jsonl.zst

# !python /content/smoothquant/examples/export_int8_model.py \
#     --model-name facebook/opt-125m \
#     --act-scales /content/smoothquant/act_scales/opt-125m.pt \
#     --output-path /content/quant_outputs/smoothquant_real_int8_export \
#     --dataset-path /content/val.jsonl.zst \
#     --num-samples 128 \
#     --seq-len 512

In [9]:
import shutil
from google.colab import files

# Define the output directory (already defined in config_cell as OUTPUT_DIR)
# OUTPUT_DIR is Path('/content/quant_outputs/smoothquant_w8a8')

output_zip_name = '/content/quant_outputs.zip'
shutil.make_archive(output_zip_name.replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f"Created {output_zip_name}")

files.download(output_zip_name)


Created /content/quant_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>